# 🛡️ SENTINEL — Gujarat Police Command & Intelligence Platform
## 🚀 1-Click Google Colab GPU Backend + Cloudflare Zero-Trust Tunnel

**100% Free Hosting Solution (Option B: Hackathon Secret Weapon)**
- **Compute**: Free NVIDIA Tesla T4 GPU (16 GB VRAM) for real-time 1 FPS keyframe YOLOv8 + OCR inference
- **Tunnel**: Cloudflare Zero-Trust Tunnel (`trycloudflare.com`) with instant public HTTPS and zero credit cards
- **Frontend**: Host your UI on Vercel or GitHub Pages, and connect dynamically via the Cloud GPU button!

---
### Step 1: Ensure GPU Acceleration is Enabled
In the Colab menu above: **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ Click **Save**.

In [ ]:
# Verify GPU availability and memory allocation
!nvidia-smi

### Step 2: Clone or Pull Sentinel Codebase
Enter your GitHub repository URL below (or use the current project bundle):

In [ ]:
import os

# Enter your GitHub repository clone URL here
REPO_URL = "https://github.com/YOUR_USERNAME/GOG_Hackathon.git"

if not os.path.exists("GOG_Hackathon") and not os.path.exists("backend"):
    print(f"[*] Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL}
    if os.path.exists("GOG_Hackathon"):
        %cd GOG_Hackathon
else:
    print("[+] Codebase directory detected.")
    if os.path.exists("GOG_Hackathon"):
        %cd GOG_Hackathon

!pwd
!ls -lh

### Step 3: Install High-Performance CUDA Dependencies
Installs PyTorch with CUDA 11.8/12.1 acceleration, Ultralytics YOLOv8, EasyOCR, FastAPI, and Uvicorn.

In [ ]:
print("[*] Installing PyTorch with CUDA support and vision libraries...")
!pip install --quiet --upgrade pip
!pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install --quiet ultralytics easyocr fastapi uvicorn python-multipart pyjwt bcrypt opencv-python-headless yt-dlp pillow requests
print("[+] All AI dependencies successfully installed!")

### Step 4: Download Cloudflare Tunnel Binary (`cloudflared`)

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

### Step 5: Launch Sentinel FastAPI Backend Server (with GPU Acceleration)

In [ ]:
import subprocess
import time
import requests

print("[*] Starting Sentinel FastAPI Backend on port 8000...")
backend_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "backend.server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait up to 10 seconds for server startup
started = False
for _ in range(20):
    time.sleep(0.5)
    try:
        r = requests.get("http://localhost:8000/health", timeout=1.0)
        if r.status_code == 200:
            started = True
            break
    except Exception:
        pass

if started:
    print("\n[+] ✅ Sentinel FastAPI Backend is ONLINE on port 8000!")
else:
    print("\n[!] Backend is still initializing or encountered an alert. Checking logs...")

### Step 6: Launch Cloudflare Tunnel & Display Public HTTPS URL
Run this cell to establish the live zero-trust tunnel and get your public URL to plug into Vercel!

In [ ]:
import subprocess
import re
import sys

print("[*] Establishing Cloudflare Zero-Trust Tunnel to http://localhost:8000...")
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1
)

tunnel_url = None
pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

for line in tunnel_proc.stderr:
    match = pattern.search(line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "=" * 75)
        print("  🚀 SENTINEL CLOUDFLARE TUNNEL IS LIVE!")
        print(f"  🔗 PUBLIC BACKEND URL:  {tunnel_url}")
        print(f"  📡 API HEALTH CHECK:    {tunnel_url}/health")
        print(f"  📚 INTERACTIVE SWAGGER: {tunnel_url}/docs")
        print("=" * 75)
        print("\n👉 HOW TO CONNECT YOUR VERCEL FRONTEND:")
        print(f"1. Open your Vercel site: https://your-site.vercel.app")
        print(f"2. Click 'Cloud GPU' button in the top header")
        print(f"3. Paste this URL: {tunnel_url}")
        print(f"4. Click Connect! You now have a 100% FREE GPU-powered CCTV AI Platform!")
        print("=" * 75 + "\n")
        break

# Keep cell alive while tunnel is active
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("\n[*] Tunnel interrupted by user.")
    tunnel_proc.terminate()